# Quality classifier (Claude Opus → Qwen2.5-0.5B) — отбор данных для SFT

Эксперимент **Шаг 5 (внешний judge)** по `step5_quality_classifier_plan.md`, обновлённый под новый no-leak baseline-протокол:

1. **Стадия A** — разметка **3000** примеров `pool_ds[60_000:63_000]` через **Anthropic Message Batches** (`claude-opus-4-7`), промпт с **prompt caching** на system.
2. **Стадия B** — обучение **base Qwen2.5-0.5B** + LoRA + регрессионная голова (MSE к score 1–5), tokenizer получает фиксированный Qwen2.5 ChatML через `get_chat_template(..., "qwen-2.5")`.
3. **Стадия C** — скоринг всего no-leak **pool_ds** (200k), кеш `scores.npy`.
4. **Стадия D** — фильтр шума, exclude `[30k, 63k)`, top-90k по quality score → `selected_indices.npy` + `ds_split`.
5. **Target** — base `Qwen2.5-1.5B` QLoRA, тот же обновлённый selection-стек, что в `selection_entropy.ipynb`.

Common val зарезервирован до построения `pool_ds`: `ds_full.shuffle(seed=SEED)[0:COMMON_VAL_HOLDOUT_SIZE]`. Поэтому финальная метрика не пересекается с annotation/scoring pool или target train.


## Шаг 1.1 — Установка зависимостей

Чистая переустановка ключевых пакетов + установка свежего `unsloth` с PyPI (он сам подберёт совместимые `transformers`, `datasets`, `trl`, `peft`, `torchao`). Жёстко пинить версии не пытаемся: сборки Colab меняются, Unsloth выкатывает новый релиз каждый месяц, и попытка зафиксировать всё ломается на следующей неделе.

После установки **обязательно** перезапустить runtime (Colab поднимет уведомление в конце ячейки). После рестарта продолжаем со следующей ячейки.

In [ ]:
# Сносим всё, что мы могли поставить кривой версии в предыдущих попытках.
# Ошибки игнорируем — пакета может и не быть.
!pip uninstall -y -q \
    transformers datasets trl peft \
    unsloth unsloth_zoo torchao \
    2>/dev/null

# Ставим unsloth с PyPI — он сам подтянет совместимые transformers/datasets/trl/peft/torchao.
!pip install --quiet --upgrade unsloth

# Дополняем тем, что unsloth не тянет, но нам нужно для SFT и квантизации.
!pip install --quiet --upgrade bitsandbytes accelerate sentencepiece

# Разметка Claude (Message Batches) + scipy для Spearman
!pip install --quiet anthropic scipy


## Шаг 1.1.5 — Проверка версий

Печатаем фактические версии установленных пакетов. От этого зависит API SFTConfig (в `trl>=0.20` `max_seq_length` переименован в `max_length`, а `dataset_text_field` депрекейтнут), а также то, какие функции Unsloth доступны.

Запускать **после** рестарта runtime, прежде чем идти дальше.

In [ ]:
import importlib.metadata as _md

print("Установленные версии:")
for pkg in [
    "unsloth", "unsloth_zoo",
    "transformers", "trl", "peft", "datasets",
    "torchao", "bitsandbytes", "accelerate",
    "torch",
]:
    try:
        v = _md.version(pkg)
    except _md.PackageNotFoundError:
        v = "MISSING"
    print(f"  {pkg:15s} {v}")

## Шаг 1.2 — Проверка железа и обязательный bf16

T4 (Turing, sm_75) **не поддерживает bf16**. A100/L4 (Ampere+, sm_80+) — поддерживает. Если bf16 недоступен, ноутбук должен остановиться: fallback на fp16 в этом эксперименте запрещён.

Не используем `torch.cuda.is_bf16_supported()`: в PyTorch есть [баг](https://github.com/pytorch/pytorch/issues/118122), из-за которого функция возвращает `True` для Turing. Проверяем напрямую через compute capability.

> **Важно:** `import unsloth` обязательно делаем здесь, до любого импорта из `transformers`. Иначе теряются оптимизации Unsloth (FastAttention, патчи torch и т.д.) и появляется warning про порядок импортов.


In [ ]:
import unsloth   # ОБЯЗАТЕЛЬНО до transformers
import torch


def detect_precision():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Для этого запуска нужен GPU с bf16 (A100/L4/H100 или другой sm_80+).")
    major, minor = torch.cuda.get_device_capability(0)
    if major < 8:
        raise RuntimeError(
            f"bf16 недоступен на GPU sm_{major}{minor}. "
            "Fallback на fp16 отключён намеренно; запусти на A100/L4/H100 или другом sm_80+."
        )
    return {"bf16": True, "fp16": False}


def check_hardware():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Для этого запуска нужен GPU с bf16 (A100/L4/H100 или другой sm_80+).")

    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    major, minor = torch.cuda.get_device_capability(0)
    p = detect_precision()

    print(f"GPU:                {name}")
    print(f"VRAM:               {vram_gb:.1f} GB")
    print(f"Compute capability: sm_{major}{minor}")
    print("Precision:          bf16 (required)")

    if vram_gb < 4:
        print("СОВЕТ: VRAM < 4 GB — переключитесь на Qwen2.5-0.5B")
    elif vram_gb < 8:
        print("OK: Qwen2.5-1.5B + 4bit поместится с запасом")
    else:
        print("OK: можно использовать Qwen2.5-1.5B или 3B")


check_hardware()


## Шаг 1.2.5 — Google Drive (для устойчивости к падению сессии)

Чекпоинты копятся каждые 200 шагов. Если сессия Colab упадёт (а она упадёт рано или поздно — таймаут, разрыв сети, OOM), всё в `/content/` пропадает. Чтобы не терять часы обучения, монтируем Google Drive и сохраняем туда.

После монтирования: даже после полного дисконнекта можно заново открыть ноутбук, прогнать все ячейки сверху вниз — `trainer.train(resume_from_checkpoint=True)` подхватит с последнего чекпоинта.

Если запускаешь локально (не Colab) или не хочешь Drive — поставь `USE_DRIVE = False`, сохранение пойдёт в текущую папку.

Для параллельных запусков: если `/content/drive` уже занят локальными файлами или другой тетрадкой, ячейка пробует запасные mountpoint-ы (`/content/gdrive`, `/content/drive_1`, ...). `force_remount=True` намеренно не используется, чтобы не сломать соседний прогон.


In [ ]:
USE_DRIVE = True   # False = сохранять локально (потеряется при разрыве сессии)

if USE_DRIVE:
    try:
        import os
        from google.colab import drive

        def _is_drive_mounted(path):
            return os.path.isdir(os.path.join(path, "MyDrive"))

        def _mount_drive_parallel_safe():
            # Не используем force_remount=True: это может отвалить Drive у другой тетрадки.
            candidates = ["/content/drive", "/content/gdrive"] + [
                f"/content/drive_{i}" for i in range(1, 8)
            ]

            for mountpoint in candidates:
                if _is_drive_mounted(mountpoint):
                    print(f"Google Drive уже доступен: {mountpoint}")
                    return mountpoint

            last_error = None
            for mountpoint in candidates:
                try:
                    if os.path.exists(mountpoint):
                        if not os.path.isdir(mountpoint):
                            print(f"{mountpoint} занят файлом — пробуем следующий")
                            continue
                        if os.listdir(mountpoint):
                            print(f"{mountpoint} занят локальными файлами — пробуем следующий")
                            continue

                    os.makedirs(mountpoint, exist_ok=True)
                    drive.mount(mountpoint)
                    if _is_drive_mounted(mountpoint):
                        return mountpoint
                    print(f"WARNING: {mountpoint} смонтирован, но MyDrive не найден; используем как есть")
                    return mountpoint
                except ValueError as exc:
                    last_error = exc
                    if _is_drive_mounted(mountpoint):
                        return mountpoint
                    print(f"{mountpoint}: {exc}")

            raise RuntimeError(
                "Не удалось подключить Google Drive: все mountpoint заняты. "
                "Очисти локальные /content/drive* или поставь USE_DRIVE=False."
            ) from last_error

        DRIVE_MOUNT = _mount_drive_parallel_safe()
        OUTPUT_BASE = f"{DRIVE_MOUNT}/MyDrive/diploma"
        print(f"OUTPUT_BASE = {OUTPUT_BASE} (Google Drive)")
    except ImportError:
        print("google.colab недоступен — сохраняем локально")
        OUTPUT_BASE = "."
else:
    OUTPUT_BASE = "."
    print(f"OUTPUT_BASE = {OUTPUT_BASE} (локально)")


## Шаг 1.3 — Конфиг

Все константы в одном месте. Между экспериментами **меняется только `STRATEGY`** — `OUTPUT_DIR` собирается из неё автоматически, поэтому артефакты разных запусков не пересекаются по папкам.

`SUBSAMPLE_SIZE = 90_000`, `NUM_EPOCHS = 1` — обновлённый target-бюджет selection-экспериментов. Он соответствует новому baseline-протоколу: вместо `30_000 × 3 эпохи` модель один раз проходит по 90k уникальных примеров.

`COMMON_VAL_HOLDOUT_SIZE` отрезается первым из `shuffle(seed=SEED)` и не попадает ни в annotation/scoring pool, ни в train.


In [ ]:
# =============================================================================
# ★ STRATEGY — Quality classifier (Claude → 0.5B), fast pool 200k
# =============================================================================
STRATEGY = "quality_classifier_90k_from_200k_noleak"
# =============================================================================

MODEL_NAME      = "Qwen/Qwen2.5-1.5B"
DATASET_NAME    = "d0rj/ru-instruct"

SUBSAMPLE_SIZE  = 90_000
POOL_LIMIT      = 200_000

MAX_SEQ_LEN     = 2048
BATCH_SIZE      = 4
GRAD_ACCUM      = 2
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 1
WARMUP_RATIO    = 0.03
SEED            = 42
VAL_SIZE        = 0.05
COMMON_VAL_HOLDOUT_SIZE = int(SUBSAMPLE_SIZE * VAL_SIZE)  # внешний common val, исключён из train/selection pools

OUTPUT_DIR      = f"{OUTPUT_BASE}/outputs/{STRATEGY}"
ADAPTER_DIR     = f"{OUTPUT_DIR}/adapter"
LOG_PATH        = f"{OUTPUT_DIR}/training_log.json"
METRICS_PATH    = f"{OUTPUT_DIR}/final_metrics.json"
INDICES_PATH    = f"{OUTPUT_DIR}/selected_indices.npy"
SCORES_PATH     = f"{OUTPUT_DIR}/scores.npy"

# ----- Quality classifier -----
ANNOTATION_START = 60_000
ANNOTATION_SIZE  = 3_000

SCORER_MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SCORER_LORA_R     = 16
SCORER_LR         = 2e-4
SCORER_EPOCHS     = 3
SCORER_BATCH_SIZE = 8

ANNOTATIONS_PATH = f"{OUTPUT_DIR}/annotations.json"
SCORER_DIR        = f"{OUTPUT_DIR}/quality_scorer"
SCORER_LOG_PATH   = f"{OUTPUT_DIR}/scorer_training_log.json"

NOISE_THRESHOLD         = 2.0
SCORE_CHECKPOINT_EVERY  = 10_000

CLAUDE_MODEL = "claude-opus-4-7"

EVAL_SAMPLES       = 500
FINAL_EVAL_SAMPLES = 200

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

if POOL_LIMIT is not None and SUBSAMPLE_SIZE > POOL_LIMIT:
    raise ValueError(f"SUBSAMPLE_SIZE={SUBSAMPLE_SIZE} больше POOL_LIMIT={POOL_LIMIT}")

print(f"\nSTRATEGY:    {STRATEGY}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


## Шаг 1.3.5 — Опционально: удалить артефакты прошлого прогона

В следующей ячейке **`RESET_PREVIOUS_RUN = True`** полностью удаляет **`OUTPUT_DIR`** стратегии (чекпоинты, `adapter/`, `scores.npy`, `annotations.json`, `batch_id.txt`, `quality_scorer/`, `selected_indices.npy`, логи — всё внутри этой папки). Нужен для **чистого** перезапуска без resume и без смешивания метрик.

**Внимание:** на Google Drive удаление необратимо.

> Нумерация ячеек ниже по файлу сдвинута на **+2** относительно старых упоминаний «Cell 12 / 16 / 24» в markdown — имей в виду при поиске по тексту ноутбука.


In [ ]:
import os
import shutil

RESET_PREVIOUS_RUN = False  # True = полностью снести OUTPUT_DIR и создать заново

if RESET_PREVIOUS_RUN:
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
        print(f"Удалено: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("Пустая OUTPUT_DIR создана.")
else:
    print("RESET_PREVIOUS_RUN=False — OUTPUT_DIR не трогаем.")


## Шаг 1.4 — Подготовка датасета + Quality classifier (стадии A–D)

Одна большая ячейка со стадиями:

- **pool_ds** — no-leak fast-pool: `ds_full.shuffle(seed=SEED)[COMMON_VAL_HOLDOUT_SIZE:COMMON_VAL_HOLDOUT_SIZE+POOL_LIMIT]`.
- **A** — Anthropic Message Batches на `pool_ds[ANNOTATION_START:ANNOTATION_START+ANNOTATION_SIZE]` → `annotations.json`.
- **B** — обучение base Qwen2.5-0.5B + LoRA + head → `quality_scorer/`.
- **C** — скоринг всего `pool_ds` → `scores.npy`.
- **D** — фильтр шума, exclude `[30k, 63k)`, top-90k → `selected_indices.npy` + `ds_split`.

Common-val примеры недоступны для разметки, скоринга и отбора по построению.


In [ ]:
import json
import os
import gc
import re
import time
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from unsloth.chat_templates import get_chat_template
from peft import LoraConfig, PeftModel, get_peft_model

# =============================================================================
# Стадии A–D: Claude batch → обучение 0.5B scorer → скоринг 200k → top-90k
# Переключатели (между сессиями Colab правь здесь):
#   1) DO_CLAUDE_SUBMIT=True  → отправить batch, сохранить batch_id.txt, STOP
#   2) DO_CLAUDE_POLL=True   → дождаться ended, записать annotations.json
#   3) TRAIN_SCORER / SCORE_POOL / BUILD_SELECTION — как обычно
# =============================================================================
DO_CLAUDE_SUBMIT = False
DO_CLAUDE_POLL = False
TRAIN_SCORER = True
SCORE_POOL = True
BUILD_SELECTION = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----- pool_ds (тот же протокол, что IFD / entropy / RHO) -----
ds_full = load_dataset(DATASET_NAME, split="train")
print(f"Полный размер ds_full: {len(ds_full)}")
ds_shuffled = ds_full.shuffle(seed=SEED)
pool_start = COMMON_VAL_HOLDOUT_SIZE
pool_end = pool_start + POOL_LIMIT
if pool_end > len(ds_shuffled):
    raise ValueError(
        f"COMMON_VAL_HOLDOUT_SIZE + POOL_LIMIT = {pool_end} больше размера датасета ({len(ds_shuffled)})"
    )
pool_ds = ds_shuffled.select(range(pool_start, pool_end))
print(
    f"pool_ds: {len(pool_ds)} из shuffle(seed={SEED})[{pool_start}:{pool_end}], "
    f"common val = [0:{COMMON_VAL_HOLDOUT_SIZE})"
)

SYSTEM_PROMPT = """Ты эксперт-оценщик качества обучающих примеров для языковой модели.
Твоя задача — оценить каждую пару (инструкция, ответ) по шкале от 1 до 5.

КРИТЕРИИ ОЦЕНКИ:
1 — Очень плохой пример: ответ некорректный, нерелевантный инструкции, бессмысленный или содержит грубые фактические/логические ошибки.
2 — Плохой пример: ответ частично некорректный, слабо связан с инструкцией, неполный или с заметными недочётами.
3 — Средний пример: ответ корректный и относится к инструкции, но банальный, слишком простой или шаблонный — модель его и так знает.
4 — Хороший пример: ответ корректный, релевантный, содержит полезную информацию или нетривиальное рассуждение. Учиться на нём имеет смысл.
5 — Отличный пример: ответ исключительно качественный — глубокий, точный, развёрнутый там где нужно. Идеальный образец для обучения.

ОБЯЗАТЕЛЬНО:
- Учитывай язык: примеры на русском, оценивай качество русского ответа.
- Оценивай ОТВЕТ как обучающий сигнал, а не как помощь конечному пользователю.
- Возвращай ТОЛЬКО JSON в формате: {"score": N, "reason": "1-2 фразы"}
- N — целое число от 1 до 5.
"""

USER_PROMPT_TEMPLATE = """Инструкция:
{instruction}

Ответ:
{response}

Оцени этот обучающий пример."""


def _extract_instruction_response(msgs):
    instruction = next((m["content"] for m in msgs if m["role"] == "user"), "")
    response_text = next((m["content"] for m in msgs[::-1] if m["role"] == "assistant"), "")
    return instruction, response_text


def format_for_scorer(instruction: str, response: str, tokenizer) -> str:
    messages = [
        {
            "role": "system",
            "content": "Ты оцениваешь качество обучающего примера. Оценка от 1 до 5.",
        },
        {
            "role": "user",
            "content": (
                f"Инструкция: {instruction}\n\nОтвет: {response}\n\nОцени качество ответа от 1 до 5."
            ),
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


def _dtype_for_scorer():
    detect_precision()  # bf16-only protocol; raises if unavailable
    return torch.bfloat16


class QualityScorer(nn.Module):
    """Qwen2.5-0.5B + LoRA + регрессионная голова → score в [1, 5]."""

    def __init__(self, base_model_name: str, lora_r: int):
        super().__init__()
        dt = _dtype_for_scorer()
        self.backbone = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            dtype=dt,
            device_map="auto",
            trust_remote_code=True,
        )
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_r,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        self.backbone = get_peft_model(self.backbone, lora_config)
        hidden_size = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1),
        )
        self.head.to(next(self.backbone.parameters()).device)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )
        last_hidden = out.hidden_states[-1]
        seq_lengths = attention_mask.sum(dim=1).long() - 1
        b = input_ids.size(0)
        batch_i = torch.arange(b, device=input_ids.device)
        last_tok = last_hidden[batch_i, seq_lengths].float()
        raw = self.head(last_tok).squeeze(-1)
        return 1.0 + 4.0 * torch.sigmoid(raw)

    def save_full(self, path: str, tokenizer):
        Path(path).mkdir(parents=True, exist_ok=True)
        self.backbone.save_pretrained(path)
        tokenizer.save_pretrained(path)
        torch.save(self.head.state_dict(), os.path.join(path, "head.pt"))

    @classmethod
    def load_for_inference(cls, base_model_name: str, path: str, lora_r: int):
        _ = lora_r  # LoRA rank зашит в adapter_config.json
        tok = get_chat_template(
            AutoTokenizer.from_pretrained(path, use_fast=True),
            chat_template="qwen-2.5",
        )
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        dt = _dtype_for_scorer()
        base = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            dtype=dt,
            device_map="auto",
            trust_remote_code=True,
        )
        backbone = PeftModel.from_pretrained(base, path)
        backbone.eval()
        obj = cls.__new__(cls)
        nn.Module.__init__(obj)
        obj.backbone = backbone
        hidden_size = backbone.config.hidden_size
        obj.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1),
        )
        dev = next(backbone.parameters()).device
        obj.head.load_state_dict(torch.load(os.path.join(path, "head.pt"), map_location=dev))
        obj.head.to(dev)
        obj.head.eval()
        return obj, tok


class AnnotationDataset(Dataset):
    def __init__(self, ann_rows, pool_ds, tokenizer, max_len=1024):
        self.rows = [a for a in ann_rows if a.get("score") is not None]
        self.pool_ds = pool_ds
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        ann = self.rows[idx]
        pool_idx = int(ann["pool_idx"])
        msgs = self.pool_ds[pool_idx]["conversations"]
        ins, resp = _extract_instruction_response(msgs)
        text = format_for_scorer(ins, resp, self.tok)
        enc = self.tok(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "score": torch.tensor(float(ann["score"]), dtype=torch.float32),
        }


def _parse_claude_result_row(row):
    """Разбор одной строки JSONL из batches.results (устойчиво к версиям SDK)."""
    pool_idx = None
    score = None
    reason = ""
    if isinstance(row, dict):
        d = row
    elif hasattr(row, "model_dump"):
        d = row.model_dump()
    else:
        d = {}
    cid = d.get("custom_id")
    if cid is None:
        cid = getattr(row, "custom_id", None)
    if cid and str(cid).startswith("pool_"):
        pool_idx = int(str(cid).split("_", 1)[1])
    res = d.get("result")
    if res is None:
        res = getattr(row, "result", None)
    if res is None:
        return pool_idx, None, "no result"
    rdict = res if isinstance(res, dict) else (res.model_dump() if hasattr(res, "model_dump") else {})
    rtype = rdict.get("type") or getattr(res, "type", None)
    if rtype != "succeeded":
        return pool_idx, None, f"ERROR type={rtype}"
    msg = rdict.get("message") or getattr(res, "message", None)
    if msg is None:
        return pool_idx, None, "no message"
    md = msg if isinstance(msg, dict) else (msg.model_dump() if hasattr(msg, "model_dump") else {})
    parts = md.get("content") or []
    text = ""
    for p in parts:
        if isinstance(p, dict) and p.get("type") == "text":
            text = p.get("text", "")
            break
        pt = getattr(p, "type", None)
        if pt == "text":
            text = getattr(p, "text", "")
            break
    text = (text or "").strip()
    try:
        parsed = json.loads(text)
        score = int(parsed["score"])
        reason = str(parsed.get("reason", ""))
    except (json.JSONDecodeError, KeyError, ValueError, TypeError):
        m = re.search(r'"score"\s*:\s*(\d+)', text)
        score = int(m.group(1)) if m else None
        reason = text[:500]
    if score is not None and not (1 <= score <= 5):
        score = None
        reason = "score out of range"
    return pool_idx, score, reason


# =============================================================================
# Стадия A: Claude Message Batches
# =============================================================================
ANNOTATION_INDICES = list(range(ANNOTATION_START, ANNOTATION_START + ANNOTATION_SIZE))
BATCH_ID_PATH = os.path.join(OUTPUT_DIR, "batch_id.txt")

annotations = None
if os.path.isfile(ANNOTATIONS_PATH):
    with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
        annotations = json.load(f)
    print(f"Загружен {ANNOTATIONS_PATH} ({len(annotations)} записей)")
elif DO_CLAUDE_POLL and os.path.isfile(BATCH_ID_PATH):
    import anthropic

    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("Нужен ANTHROPIC_API_KEY в окружении для poll.")
    client = anthropic.Anthropic()
    with open(BATCH_ID_PATH, "r", encoding="utf-8") as f:
        batch_id = f.read().strip()

    def _wait_batch(bid, poll_interval=60):
        while True:
            b = client.messages.batches.retrieve(bid)
            st = getattr(b, "processing_status", None)
            st = str(st) if st is not None else ""
            rc = getattr(b, "request_counts", None)
            if rc is not None:
                proc = getattr(rc, "processing", 0) or 0
                succ = getattr(rc, "succeeded", 0) or 0
                err = getattr(rc, "errored", 0) or 0
                done = succ + err
                tot = proc + succ + err
                print(
                    f"[{time.strftime('%H:%M:%S')}] status={st} done={done}/{tot} err={err}",
                )
            else:
                print(f"[{time.strftime('%H:%M:%S')}] status={st}")
            if st in ("canceled", "cancelled"):
                raise RuntimeError(f"Batch {bid} отменён (status={st})")
            if st in ("ended", "completed"):
                return b
            time.sleep(poll_interval)

    _wait_batch(batch_id)
    results_list = []
    stream = client.messages.batches.results(batch_id)
    for line in stream:
        if isinstance(line, bytes):
            row = json.loads(line.decode())
        elif isinstance(line, str):
            row = json.loads(line)
        else:
            row = line
        pi, sc, rs = _parse_claude_result_row(row)
        if pi is not None:
            results_list.append({"pool_idx": pi, "score": sc, "reason": rs})
    with open(ANNOTATIONS_PATH, "w", encoding="utf-8") as f:
        json.dump(results_list, f, ensure_ascii=False, indent=2)
    annotations = results_list
    print(f"Сохранено {ANNOTATIONS_PATH} ({len(annotations)})")
elif DO_CLAUDE_SUBMIT:
    import anthropic

    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("Нужен ANTHROPIC_API_KEY в окружении для submit.")
    client = anthropic.Anthropic()
    requests = []
    for idx in ANNOTATION_INDICES:
        msgs = pool_ds[idx]["conversations"]
        instruction, response_text = _extract_instruction_response(msgs)
        user_prompt = USER_PROMPT_TEMPLATE.format(
            instruction=instruction[:2000],
            response=response_text[:3000],
        )
        requests.append(
            {
                "custom_id": f"pool_{idx}",
                "params": {
                    "model": CLAUDE_MODEL,
                    "max_tokens": 150,
                    "system": [
                        {
                            "type": "text",
                            "text": SYSTEM_PROMPT,
                            "cache_control": {"type": "ephemeral"},
                        },
                    ],
                    "messages": [{"role": "user", "content": user_prompt}],
                },
            },
        )
    batch = client.messages.batches.create(requests=requests)
    bid = getattr(batch, "id", None)
    with open(BATCH_ID_PATH, "w", encoding="utf-8") as f:
        f.write(str(bid))
    print(f"Batch создан: id={bid} → {BATCH_ID_PATH}")
    raise RuntimeError(
        "Остановка после submit: дождись ended на Anthropic, затем "
        "DO_CLAUDE_SUBMIT=False, DO_CLAUDE_POLL=True и перезапусти ячейку.",
    )
else:
    raise RuntimeError(
        "Нет annotations.json. Либо DO_CLAUDE_SUBMIT=True (отправить batch), "
        "либо DO_CLAUDE_POLL=True при наличии batch_id.txt, либо положи файл вручную.",
    )

valid_ann = [a for a in annotations if a.get("score") is not None]
print(f"Валидных разметок: {len(valid_ann)} / {len(annotations)}")
if len(valid_ann) < int(ANNOTATION_SIZE * 0.85):
    print("WARNING: много пустых score — проверь промпт / парсинг JSONL.")

# Распределение скоров Claude
_dist = Counter(a["score"] for a in valid_ann)
print("\nРаспределение скоров (Claude):")
for s in sorted(_dist):
    print(f"  {s}: {_dist[s]} ({100 * _dist[s] / max(len(valid_ann), 1):.1f}%)")

# =============================================================================
# Стадия B: обучение scorer
# =============================================================================
scorer_tok = get_chat_template(
    AutoTokenizer.from_pretrained(SCORER_MODEL_NAME, use_fast=True),
    chat_template="qwen-2.5",
)
if scorer_tok.pad_token_id is None:
    scorer_tok.pad_token = scorer_tok.eos_token
scorer_tok.padding_side = "right"

if TRAIN_SCORER:
    random.Random(SEED).shuffle(valid_ann)
    n_val = max(1, int(len(valid_ann) * 0.1))
    val_rows = valid_ann[:n_val]
    train_rows = valid_ann[n_val:]
    print(f"Scorer train={len(train_rows)} val={len(val_rows)}")

    train_ds = AnnotationDataset(train_rows, pool_ds, scorer_tok, max_len=1024)
    val_ds = AnnotationDataset(val_rows, pool_ds, scorer_tok, max_len=1024)
    train_loader = DataLoader(train_ds, batch_size=SCORER_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=SCORER_BATCH_SIZE, shuffle=False)

    model = QualityScorer(SCORER_MODEL_NAME, SCORER_LORA_R)
    model.train()
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=SCORER_LR,
        weight_decay=0.01,
    )
    total_steps = max(1, len(train_loader) * SCORER_EPOCHS)
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=max(1, int(total_steps * 0.1)),
        num_training_steps=total_steps,
    )
    loss_fn = nn.MSELoss()
    log_epochs = []

    for epoch in range(SCORER_EPOCHS):
        model.train()
        tr_losses = []
        for batch in train_loader:
            dev = next(model.backbone.parameters()).device
            input_ids = batch["input_ids"].to(dev)
            attention_mask = batch["attention_mask"].to(dev)
            y = batch["score"].to(dev)
            pred = model(input_ids, attention_mask)
            loss = loss_fn(pred, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        preds, truths = [], []
        with torch.no_grad():
            for batch in val_loader:
                dev = next(model.backbone.parameters()).device
                input_ids = batch["input_ids"].to(dev)
                attention_mask = batch["attention_mask"].to(dev)
                y = batch["score"].to(dev)
                pred = model(input_ids, attention_mask)
                va_losses.append(loss_fn(pred, y).item())
                preds.extend(pred.detach().float().cpu().tolist())
                truths.extend(y.detach().float().cpu().tolist())

        mae = float(np.mean([abs(p - t) for p, t in zip(preds, truths)]))
        sp = None
        try:
            from scipy.stats import spearmanr

            sp, _ = spearmanr(preds, truths)
        except Exception:
            pass
        rec = {
            "epoch": epoch + 1,
            "train_mse": float(np.mean(tr_losses)),
            "val_mse": float(np.mean(va_losses)),
            "val_mae": mae,
            "val_spearman": None if sp is None or np.isnan(sp) else float(sp),
        }
        log_epochs.append(rec)
        sp_s = f"{rec['val_spearman']:+.3f}" if rec["val_spearman"] is not None else "n/a"
        print(
            f"Epoch {epoch + 1}/{SCORER_EPOCHS} train_mse={rec['train_mse']:.4f} "
            f"val_mse={rec['val_mse']:.4f} val_mae={mae:.3f} val_Spearman={sp_s}",
        )

    with open(SCORER_LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(log_epochs, f, indent=2, ensure_ascii=False)
    model.save_full(SCORER_DIR, scorer_tok)
    print(f"Scorer сохранён: {SCORER_DIR}")
    del model, opt, sched, train_loader, val_loader, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =============================================================================
# Стадия C: скоринг pool
# =============================================================================
if SCORE_POOL:
    if not os.path.isdir(SCORER_DIR) or not os.path.isfile(os.path.join(SCORER_DIR, "head.pt")):
        raise RuntimeError(f"Нет обученного scorer в {SCORER_DIR} — сначала TRAIN_SCORER=True.")
    inf_model, inf_tok = QualityScorer.load_for_inference(SCORER_MODEL_NAME, SCORER_DIR, SCORER_LORA_R)
    inf_model.eval()
    dev = next(inf_model.parameters()).device

    def load_partial_scores(path, n_total):
        if os.path.exists(path):
            scores = np.load(path)
            if len(scores) != n_total:
                raise RuntimeError("Размер кеша scores ≠ pool — удали файл.")
            n_done = int((~np.isnan(scores)).sum())
            print(f"Кеш scores: {n_done}/{n_total}")
            return scores
        return np.full(n_total, np.nan, dtype=np.float32)

    scores_q = load_partial_scores(SCORES_PATH, len(pool_ds))
    todo = np.where(np.isnan(scores_q))[0].tolist()
    bs = 16
    pbar = tqdm(total=len(todo), desc=f"Quality scores (batch={bs})")
    since = 0
    for b0 in range(0, len(todo), bs):
        idxs = todo[b0 : b0 + bs]
        texts = []
        for i in idxs:
            msgs = pool_ds[int(i)]["conversations"]
            ins, resp = _extract_instruction_response(msgs)
            texts.append(format_for_scorer(ins, resp, inf_tok))
        enc = inf_tok(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )
        enc = {k: v.to(dev) for k, v in enc.items()}
        try:
            with torch.no_grad():
                pred = inf_model(enc["input_ids"], enc["attention_mask"])
            for ii, p in zip(idxs, pred.detach().float().cpu().tolist()):
                scores_q[ii] = float(p)
        except Exception as e:
            print(f"\n[!] batch {idxs[0]}: {e}")
            for ii in idxs:
                scores_q[ii] = float("nan")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        pbar.update(len(idxs))
        since += len(idxs)
        if since >= SCORE_CHECKPOINT_EVERY:
            np.save(SCORES_PATH, scores_q)
            since = 0
    pbar.close()
    np.save(SCORES_PATH, scores_q)
    nan_n = int(np.isnan(scores_q).sum())
    print(f"Скоринг готово. NaN: {nan_n}/{len(scores_q)} ({100 * nan_n / max(len(scores_q), 1):.2f}%)")
    del inf_model, inf_tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =============================================================================
# Стадия D: отбор + ds_split (как template)
# =============================================================================
if BUILD_SELECTION:
    scores_quality = np.load(SCORES_PATH)
    valid_mask = ~np.isnan(scores_quality)
    non_noise = scores_quality >= NOISE_THRESHOLD
    selectable = valid_mask & non_noise
    EXCLUDE_INDICES = set(range(30_000, 63_000))
    exclude_mask = np.array([i not in EXCLUDE_INDICES for i in range(len(scores_quality))], dtype=bool)
    selectable = selectable & exclude_mask

    n_valid = int(valid_mask.sum())
    n_after_noise = int((valid_mask & non_noise).sum())
    n_excluded_service = int((valid_mask & non_noise & ~exclude_mask).sum())
    n_sel = int(selectable.sum())
    print(f"\nВалидных скоров: {n_valid}/{len(scores_quality)}")
    print(f"После score>={NOISE_THRESHOLD}: {n_after_noise}")
    print(f"После exclude[30k:63k) в pool_ds-координатах: {n_sel}")
    if n_valid:
        print(f"Шум score<{NOISE_THRESHOLD}: {n_valid - n_after_noise}")
        print(f"Исключено служебным диапазоном: {n_excluded_service}")

    nv = max(n_valid, 1)
    print("\nБакеты по pool (доля от валидных):")
    for low, high in [(1.0, 2.0), (2.0, 3.0), (3.0, 4.0), (4.0, 5.01)]:
        bm = valid_mask & (scores_quality >= low) & (scores_quality < high)
        n = int(bm.sum())
        print(f"  [{low:.1f}, {high:.1f}): {n} ({100 * n / nv:.1f}%)")

    if n_sel < SUBSAMPLE_SIZE:
        raise RuntimeError(f"Selectable {n_sel} < SUBSAMPLE_SIZE={SUBSAMPLE_SIZE}")

    sel_idx = np.where(selectable)[0]
    top_local = np.argsort(-scores_quality[sel_idx])[:SUBSAMPLE_SIZE]
    selected_indices = np.sort(sel_idx[top_local])
    top_sc = scores_quality[selected_indices]
    print(f"\nTop-{SUBSAMPLE_SIZE}: min={top_sc.min():.3f} max={top_sc.max():.3f} mean={top_sc.mean():.3f}")

    try:
        import scipy.stats

        diag = np.where(valid_mask)[0][:5000]
        cl = np.array(
            [sum(len(m["content"]) for m in pool_ds[int(i)]["conversations"]) for i in diag],
        )
        corr, _ = scipy.stats.spearmanr(cl, scores_quality[diag])
        print(f"\nSpearman(quality_score, char_len) на 5000 валидных: {corr:+.3f}")
    except Exception as e:
        print(f"(Spearman пропущен: {e})")

    assert len(selected_indices) == SUBSAMPLE_SIZE
    assert len(set(selected_indices.tolist())) == SUBSAMPLE_SIZE
    np.save(INDICES_PATH, selected_indices)
    print(f"\nСохранено {INDICES_PATH}")

    tok_render = get_chat_template(
        AutoTokenizer.from_pretrained(MODEL_NAME),
        chat_template="qwen-2.5",
    )

    def _render(examples):
        texts = [
            tok_render.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
            for c in examples["conversations"]
        ]
        return {"text": texts}

    ds = pool_ds.select(selected_indices.tolist())
    ds = ds.map(_render, batched=True, remove_columns=ds.column_names, desc="render chat template")
    ds_split = ds.train_test_split(test_size=VAL_SIZE, seed=SEED)
    print(f"\nTrain: {len(ds_split['train'])}, Val: {len(ds_split['test'])}")

    sample_n = min(1000, len(ds_split["train"]))
    lengths = [
        len(tok_render(ex["text"], add_special_tokens=False)["input_ids"])
        for ex in ds_split["train"].select(range(sample_n))
    ]
    too_long = sum(1 for ell in lengths if ell > MAX_SEQ_LEN)
    stats = {
        "avg_len_tokens": round(sum(lengths) / len(lengths), 1),
        "max_len_tokens": max(lengths),
        "pct_over_max_seq_len": round(too_long / len(lengths) * 100, 2),
        "expected_pack_ratio": round(MAX_SEQ_LEN / (sum(lengths) / len(lengths)), 2),
    }
    print("\nСтатистика длин (выборка 1000):")
    print(json.dumps(stats, indent=2, ensure_ascii=False))
    print("\n--- Пример текста ---")
    print(ds_split["train"][0]["text"][:500])

print("\n★ STRATEGY BLOCK (quality classifier) завершён.")


## Шаг 1.5 — Загрузка модели + LoRA

QLoRA: модель в 4-bit, LoRA-адаптер сверху. По плану:
- `r=16`, `lora_alpha=16`
- Целевые модули — все линейные слои attention и MLP
- `lora_dropout=0` обязательно при 4-bit
- `use_gradient_checkpointing="unsloth"` экономит ещё ~30% VRAM (Unsloth-овская реализация быстрее transformers-native)

После загрузки модели прогоняем `tokenizer` через `get_chat_template(..., chat_template="qwen-2.5")` — фиксируем тот же ChatML-шаблон Qwen2.5, который использовался в Cell 12 при скоринге и рендере `text`. Это важно для `train_on_responses_only`: маркеры `<|im_start|>user\n` и `<|im_start|>assistant\n` должны совпадать с тем, что реально лежит в input_ids.

**Чекпоинт:** доля обучаемых параметров — ожидаем 1–2%.


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template


def load_model_with_lora():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )

    # Фиксируем Unsloth-овский ChatML-шаблон Qwen2.5 — гарантия согласованности
    # маркеров между prepare_data, тренировкой и инференсом
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Обучаемых параметров: {trainable:,} ({100 * trainable / total:.2f}%)")

    return model, tokenizer


model, tokenizer = load_model_with_lora()

## Шаг 2.1 — Обучение с `packing=True` + `train_on_responses_only`

Этот стек **зафиксирован** на обновлённом baseline/entropy-этапе и одинаков для всех экспериментов с data selection. Менять параметры здесь — значит ломать сравнимость с `baseline_masked_packed.ipynb` и другими стратегиями.

- 1 эпоха на ~85 500 train-примерах из отобранных 90k, **эффективный batch = 8** (`BATCH_SIZE=4 × GRAD_ACCUM=2`).
- `MAX_SEQ_LEN = 2048`. На `d0rj/ru-instruct` обрезается ~2–3% длинных примеров.
- **`packing=True` / `eval_packing=False`** передаются в конструктор `SFTConfig` (trl≥0.24); setattr после создания конфига мог не подхватиться в `_prepare_dataset` — см. step4_rho_fix_proposal.
- **`train_on_responses_only`** вызывается **после** `SFTTrainer(...)`. Маскирует prompt-токены под `-100`. Корректно обрабатывает несколько пар маркеров в одной упакованной последовательности (Unsloth discussion #2828).
- `warmup_ratio=0.03` вместо фиксированных `warmup_steps=500`: warmup масштабируется от реального числа шагов.
- **Не дублируем** `gradient_checkpointing=True` в `SFTConfig` — Unsloth уже включил его в `get_peft_model(use_gradient_checkpointing="unsloth")`.
- **Не используем** `assistant_only_loss=True` — несовместим с pre-rendered text.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

precision = detect_precision()
print("Используем bf16")

eval_subset = ds_split["test"].select(range(min(EVAL_SAMPLES, len(ds_split["test"]))))
print(f"Train: {len(ds_split['train'])}, in-loop eval: {len(eval_subset)}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    # gradient_checkpointing уже включён через use_gradient_checkpointing="unsloth"
    # в FastLanguageModel.get_peft_model().
    optim="adamw_8bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,

    bf16=precision["bf16"],
    fp16=precision["fp16"],

    max_length=MAX_SEQ_LEN,

    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",

    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    seed=SEED,
    data_seed=SEED,

    # trl>=0.24: packing в конструкторе (иначе setattr после SFTConfig может не попасть в _prepare_dataset)
    packing=True,
    eval_packing=False,
    dataset_num_proc=1,
)

import multiprocessing as _mp_unsloth_tgt

_unsloth_tgt_cpu = _mp_unsloth_tgt.cpu_count
_mp_unsloth_tgt.cpu_count = lambda: 1
try:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        args=training_args,
        train_dataset=ds_split["train"],
        eval_dataset=eval_subset,
    )

    # Маскируем prompt-токены: labels = -100 для всего, кроме ответа ассистента.
    # С packing в одной упакованной последовательности будет несколько пар маркеров
    # user/assistant — train_on_responses_only в свежих версиях Unsloth (см.
    # discussion #2828) корректно обрабатывает все пары.
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
finally:
    _mp_unsloth_tgt.cpu_count = _unsloth_tgt_cpu


### Sanity check: packing + маскирование

Проверяем три вещи:
1. **Packing работает**: `seq_length` в батче должна быть близка к `MAX_SEQ_LEN`, а **не** ~600 (средняя длина одного примера). Если seq_length маленький — packing не включился.
2. **Несколько assistant-сегментов в одной упакованной последовательности**: считаем число «контигуальных групп» токенов с `label != -100`. Если packing работает + маска работает, ожидаем 3–5 сегментов в каждой packed-строке.
3. **Trained-токены — это и правда ответы ассистента**: декодим их и читаем глазами. Trained-текст — конкатенация всех ответов в упакованной последовательности, разделённых `<|im_end|>`.

**Когда стоит начать переживать:**
- `seq_length` в батче ≈ 200–600 → **packing не сработал**, проверь `training_args.packing = True` и версию Unsloth (нужна свежая, см. PR #3566).
- `aggregate_ratio < 1%` или `> 95%` — `train_on_responses_only` не пробил маркеры. Скорее всего, расхождение между фиксированным Qwen2.5 ChatML-рендером (`get_chat_template(..., "qwen-2.5")`) и маркерами `<|im_start|>user\n` / `<|im_start|>assistant\n`. В этом случае посмотри декодинг начала первой последовательности — что именно там стоит вместо ожидаемого маркера.
- `assistant segments per packed sequence == 1` — несмотря на packing, маркер ищется только по первой паре. Нужна более новая версия Unsloth.
- В декодинге trained-токенов попадаются куски user-промптов или русский от лица user-а — chat template неправильный.

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
input_ids_batch = batch["input_ids"]
labels_batch    = batch["labels"]
B, L = input_ids_batch.shape
print(f"Batch: {B} packed-последовательностей × {L} токенов")
print(f"padding_side: {tokenizer.padding_side}")

# 1) Packing работает? seq_length должна быть близка к MAX_SEQ_LEN.
print(f"\n--- (1) Packing check ---")
print(f"L = {L}, MAX_SEQ_LEN = {MAX_SEQ_LEN}")
if L < MAX_SEQ_LEN * 0.7:
    print(f"WARNING: L = {L} сильно меньше MAX_SEQ_LEN — packing, похоже, НЕ сработал.")
    print("        Проверь training_args.packing = True и что Unsloth поддерживает packing.")
else:
    print(f"OK: L близок к MAX_SEQ_LEN — packing работает.")

# 2) Сколько assistant-сегментов в каждой последовательности.
#    Считаем как число «контигуальных групп токенов с label != -100».
print(f"\n--- (2) Assistant segments per packed sequence ---")
for i in range(B):
    labels = labels_batch[i]
    mask   = labels != -100
    # Число «переходов» 0→1 в маске = число assistant-сегментов.
    transitions = ((mask[1:].int() - mask[:-1].int()) == 1).sum().item()
    if mask[0]:
        transitions += 1
    print(f"  [{i}] assistant segments: {transitions}")

# 3) Per-sequence статистика + декодинг.
print(f"\n--- (3) Per-sequence ratio + decoded trained tokens ---")
for i in range(B):
    input_ids = input_ids_batch[i]
    labels    = labels_batch[i]
    mask      = labels != -100
    n_masked  = (~mask).sum().item()
    n_trained = mask.sum().item()
    ratio_i   = n_trained / L

    trained_text = tokenizer.decode(input_ids[mask], skip_special_tokens=False)
    print(f"  [{i}] masked={n_masked:5d}  trained={n_trained:5d}  ratio={ratio_i:6.2%}")
    print(f"      trained → {trained_text[:160]!r}")

# 4) Агрегированный ratio по всему батчу.
total_trained = (labels_batch != -100).sum().item()
total_tokens  = labels_batch.numel()
agg_ratio     = total_trained / total_tokens
print(f"\n--- (4) Aggregate ---")
print(f"trained_tokens = {total_trained}/{total_tokens} → ratio = {agg_ratio:.2%}")

if agg_ratio < 0.01:
    print("WARNING: маска НЕ сработала — train_on_responses_only не пробил маркер.")
    print("Декодинг начала первой последовательности (для отладки):")
    print(repr(tokenizer.decode(input_ids_batch[0][:120])))
    print("Сравни с ожидаемым '<|im_start|>user\\n...'")
elif agg_ratio > 0.95:
    print("WARNING: маска НЕ сработала — почти всё попадает в loss.")
    print("Маркер instruction_part='<|im_start|>user\\n' видимо не нашёлся.")
else:
    print("OK: маскирование работает корректно.")

In [ ]:
import os

ckpt_dirs = []
if os.path.isdir(OUTPUT_DIR):
    ckpt_dirs = sorted(d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-"))

if ckpt_dirs:
    print(f"Найдено {len(ckpt_dirs)} чекпоинт(ов): {ckpt_dirs}")
    print(f"Продолжаем с последнего: {ckpt_dirs[-1]}")
    trainer.train(resume_from_checkpoint=True)
else:
    print("Чекпоинтов нет — обучение с нуля.")
    trainer.train()

In [ ]:
with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2, ensure_ascii=False)

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Адаптер: {ADAPTER_DIR}")
print(f"Логи:    {LOG_PATH}")
print(f"Шагов сделано: {trainer.state.global_step}")

## Шаг 2.2 — Оценка качества (perplexity)

Загружаем **сохранённый адаптер**, а не свежую модель с нулевыми LoRA-весами.

Загрузку делаем через `FastLanguageModel.from_pretrained(ADAPTER_DIR, ...)`: Unsloth увидит `adapter_config.json`, сам подтянет базовую модель в 4-bit и навесит LoRA. Использовать «голый» `AutoModelForCausalLM.from_pretrained` нельзя — после `import unsloth` (ячейка 6) в `transformers` глобально пропатчен `Qwen2Attention.forward`, который ждёт атрибут `apply_qkv`.

`FastLanguageModel.for_inference(model)` дополнительно ускоряет инференс (отключает gradient checkpointing, переключает в eval).

Метрика — **perplexity по полному тексту примера**. В этом ноутбуке `text` уже отрендерен на этапе подготовки данных (Cell 12) через явно зафиксированный Qwen2.5 ChatML-шаблон. Для сравнимости другие стратегии должны использовать тот же рендер и тот же common val.

> Перед загрузкой освобождаем VRAM от обучающей копии модели/тренера через `del trainer, model` + `torch.cuda.empty_cache()`.


In [ ]:
import json
import math, time, gc

# Освобождаем VRAM от обучающей копии модели/тренера
del trainer, model
gc.collect()
torch.cuda.empty_cache()


def load_trained_model():
    model, tok = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIR,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tok


def compute_perplexity(model, tok, texts):
    losses = []
    for text in texts:
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        losses.append(loss.item())
    return math.exp(sum(losses) / len(losses))


eval_model, eval_tok = load_trained_model()

n_eval = min(FINAL_EVAL_SAMPLES, len(ds_split["test"]))
# text уже отрендерен в Cell 12 через явно зафиксированный Qwen2.5 ChatML-шаблон.
texts = [ex["text"] for ex in ds_split["test"].select(range(n_eval))]

t0 = time.time()
ppl = compute_perplexity(eval_model, eval_tok, texts)
elapsed = time.time() - t0

metrics = {
    "strategy":          STRATEGY,
    "perplexity":        round(ppl, 4),
    "eval_time_sec":     round(elapsed, 2),
    "samples_evaluated": n_eval,
    "adapter":           ADAPTER_DIR,
    "model":             MODEL_NAME,
    "subsample_size":    SUBSAMPLE_SIZE,
    "training_config":   f"packing=True, train_on_responses_only, subsample={SUBSAMPLE_SIZE}, epochs={NUM_EPOCHS}, warmup_ratio={WARMUP_RATIO}",
    # NB: это full-text PPL. Verdict ниже строится по assistant-only PPL на common val.
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Perplexity:    {ppl:.4f}")
print(f"Время:         {elapsed:.1f} сек на {n_eval} примерах")
print(f"Метрики:       {METRICS_PATH}")

## Шаг 2.3 — Сравнение с `baseline_masked_packed_base_90k_1ep_noleak` на `assistant_only_PPL`

**Зачем нужна отдельная метрика.** Full-text PPL (Cell 22) — методологически неконсистентная метрика для masked-моделей (training distribution mismatch с prompt-токенами). `assistant_only_PPL` считает CE-loss только на тех токенах, на которых модель училась через `train_on_responses_only` — корректно для всех стратегий из этого workflow.

**КРИТИЧЕСКИ ВАЖНО — общий val.** Каждая стратегия отбора создаёт свой `ds_split["test"]` из своих же отобранных данных. Распределения собственных val у разных стратегий разные, поэтому PPL на собственном val между стратегиями **несравнима**.

Решение: для сравнения с baseline используем **общий внешний val-сет**: `shuffle(seed=SEED)[0:COMMON_VAL_HOLDOUT_SIZE]`. Все train/selection/scoring pools начинаются после этого holdout, поэтому common val не попадает в обучение ни у baseline, ни у selection-стратегий.

PPL на собственном val тоже считаем — как диагностику обучения (пишется как `perplexity_assistant_only_on_own_val`), но **не** используется для verdict. PPL на общем val записывается как `perplexity_assistant_only_on_common_val` и идёт в `comparison_to_baseline`.

Источник числа baseline: `outputs/baseline_masked_packed_base_90k_1ep_noleak/final_metrics.json`. Старый fallback `2.4891` удалён: он относится к Instruct + `30k × 3` и после обновления протокола невалиден.

**Verdict с порогом шума 0.03** — временный практический порог из старого baseline-этапа. Если появятся повторные прогоны нового baseline, лучше заменить его на эмпирическую дисперсию нового протокола.


In [ ]:
import math, json, torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer


def compute_assistant_only_perplexity(model, tok, examples, max_length=MAX_SEQ_LEN):
    """PPL только на assistant-токенах. Принимает примеры в любом из форматов:
    - 'text' (предотрендеренный chat template — как в этом ноутбуке);
    - 'messages' / 'conversations' (raw conversational).
    Маркеры — ChatML-формат Qwen2.5 (<|im_start|>user\n / <|im_start|>assistant\n).
    """
    user_marker_ids = tok("<|im_start|>user\n",      add_special_tokens=False)["input_ids"]
    asst_marker_ids = tok("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]

    total_loss   = 0.0
    total_tokens = 0
    skipped      = 0

    for ex in examples:
        if "text" in ex:
            text = ex["text"]
        elif "messages" in ex:
            text = tok.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False,
            )
        elif "conversations" in ex:
            text = tok.apply_chat_template(
                ex["conversations"], tokenize=False, add_generation_prompt=False,
            )
        else:
            raise ValueError(f"Жду 'text', 'messages' или 'conversations', есть: {list(ex.keys())}")

        inputs = tok(text, return_tensors="pt", truncation=True,
                     max_length=max_length).to(model.device)
        input_ids = inputs["input_ids"][0]

        labels = input_ids.clone()
        labels[:] = -100  # маскируем всё...
        ids_list = input_ids.tolist()
        i = 0
        while i < len(ids_list):
            if ids_list[i:i + len(asst_marker_ids)] == asst_marker_ids:
                start = i + len(asst_marker_ids)
                end = len(ids_list)
                for j in range(start, len(ids_list) - len(user_marker_ids) + 1):
                    if ids_list[j:j + len(user_marker_ids)] == user_marker_ids:
                        end = j
                        break
                labels[start:end] = input_ids[start:end]   # ...и размаскируем assistant-сегменты
                i = end
            else:
                i += 1

        n_assistant = (labels != -100).sum().item()
        if n_assistant == 0:
            skipped += 1
            continue

        with torch.no_grad():
            out = model(input_ids=input_ids.unsqueeze(0), labels=labels.unsqueeze(0))

        total_loss   += out.loss.item() * n_assistant
        total_tokens += n_assistant

    if total_tokens == 0:
        raise RuntimeError("Не нашлось ни одного assistant-токена. Проверь маркеры/chat template.")
    if skipped:
        print(f"  WARNING: пропущено {skipped} примеров без assistant-сегментов (truncation?)")
    print(f"  Усреднено по {total_tokens} assistant-токенам, {len(examples) - skipped} примеров")
    return math.exp(total_loss / total_tokens)


# --- (1) PPL на СОБСТВЕННОМ val — диагностика, НЕ для сравнения между стратегиями.
n_eval = min(FINAL_EVAL_SAMPLES, len(ds_split["test"]))
own_val = ds_split["test"].select(range(n_eval))
print(f"--- PPL на СОБСТВЕННОМ val ({len(own_val)} примеров, диагностика) ---")
ppl_asst_own = compute_assistant_only_perplexity(eval_model, eval_tok, own_val)
print(f"  → {ppl_asst_own:.4f}")


# --- (2) PPL на ОБЩЕМ val — единственная метрика, по которой стратегии сравнимы.
print(
    f"\nПостроение ОБЩЕГО val (reserved {COMMON_VAL_HOLDOUT_SIZE:,}, seed=SEED; "
    "исключён из train/selection pools)..."
)
common_tok = get_chat_template(
    AutoTokenizer.from_pretrained(MODEL_NAME),
    chat_template="qwen-2.5",
)
ds_full = load_dataset(DATASET_NAME, split="train")
ds_common = ds_full.shuffle(seed=SEED).select(range(COMMON_VAL_HOLDOUT_SIZE))


def _render_common(examples):
    texts = [
        common_tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}


ds_common = ds_common.map(
    _render_common, batched=True,
    remove_columns=ds_common.column_names,
    desc="render common val",
)
common_val_size = min(FINAL_EVAL_SAMPLES, len(ds_common))
common_val = ds_common.select(range(common_val_size))

print(f"\n--- PPL на ОБЩЕМ val ({len(common_val)} примеров, главная метрика) ---")
ppl_asst_common = compute_assistant_only_perplexity(eval_model, eval_tok, common_val)
print(f"  → {ppl_asst_common:.4f}")


# --- (3) Сравнение с baseline на ОБЩЕМ val.
BASELINE_METRICS_PATH = f"{OUTPUT_BASE}/outputs/baseline_masked_packed_base_90k_1ep_noleak/final_metrics.json"
NOISE_FLOOR           = 0.03

try:
    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Не найден baseline metrics: {BASELINE_METRICS_PATH}. "
        "Сначала прогони обновлённый baseline_masked_packed.ipynb."
    ) from exc

if "perplexity_assistant_only" not in baseline:
    raise KeyError(f"В {BASELINE_METRICS_PATH} нет ключа 'perplexity_assistant_only'.")

baseline_ppl_asst = baseline["perplexity_assistant_only"]
baseline_source = BASELINE_METRICS_PATH

delta_abs = ppl_asst_common - baseline_ppl_asst
delta_pct = (ppl_asst_common / baseline_ppl_asst - 1) * 100

if delta_abs < -NOISE_FLOOR:
    verdict = f"{STRATEGY} ЛУЧШЕ baseline_masked_packed_base_90k_1ep_noleak (статзначимо)"
elif abs(delta_abs) <= NOISE_FLOOR:
    verdict = f"статистически неотличимо от baseline (|Δ| ≤ {NOISE_FLOOR} — шум)"
else:
    verdict = f"{STRATEGY} ХУЖЕ baseline_masked_packed_base_90k_1ep_noleak (статзначимо)"


# --- (4) Запись итоговых метрик.
with open(METRICS_PATH) as f:
    metrics = json.load(f)
metrics["perplexity_assistant_only_on_own_val"]    = round(ppl_asst_own, 4)
metrics["perplexity_assistant_only_on_common_val"] = round(ppl_asst_common, 4)
metrics["comparison_to_baseline"] = {
    "val_set":            f"reserved common val (shuffle(seed=SEED)[0:{COMMON_VAL_HOLDOUT_SIZE}], first {len(common_val)} examples)",
    "baseline_strategy":  "random_90000 (baseline_masked_packed_base_90k_1ep_noleak)",
    "baseline_ppl_asst":  round(baseline_ppl_asst, 4),
    "baseline_source":    baseline_source,
    "strategy_ppl_asst":  round(ppl_asst_common, 4),
    "delta_absolute":     round(delta_abs, 4),
    "delta_percent":      round(delta_pct, 2),
    "verdict":            verdict,
    "noise_floor":        NOISE_FLOOR,
    "samples_evaluated":  len(common_val),
    "note": (
        "PPL on OWN val is for diagnostics only — strategies have differently-"
        "distributed own vals, so own-val PPL is incomparable between strategies. "
        "The verdict uses COMMON val reserved before any train/selection/scoring pool."
    ),
}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 78)
print(f"  Сравнение {STRATEGY} с baseline_masked_packed_base_90k_1ep_noleak на ОБЩЕМ val")
print("=" * 78)
print(f"  baseline:               ppl_asst = {baseline_ppl_asst:.4f}")
print(f"  {STRATEGY[:26]:26s}  ppl_asst = {ppl_asst_common:.4f}  (на общем val)")
print(f"                                         {ppl_asst_own:.4f}  (на собственном val, diag)")
print(f"  Δ:                      {delta_abs:+.4f}  ({delta_pct:+.2f}%)")
print(f"  noise:                  ±{NOISE_FLOOR}")
print(f"  verdict:                {verdict}")
print("=" * 78)
print(f"\nМетрики обновлены: {METRICS_PATH}")


## Сводная таблица: baseline, entropy, RHO, IFD, quality classifier

`perplexity_assistant_only_on_common_val` из `final_metrics.json` no-leak-стратегий + запись `comparison_strategies` в текущий `METRICS_PATH`.


In [ ]:
import json

strategy_names = [
    "baseline_masked_packed_base_90k_1ep_noleak",
    "entropy_90k_from_200k_noleak",
    "loss_rho_90k_from_200k_noleak",
    "ifd_90k_from_200k_noleak",
    "quality_classifier_90k_from_200k_noleak",
]

results = {}
for s in strategy_names:
    path = f"{OUTPUT_BASE}/outputs/{s}/final_metrics.json"
    try:
        with open(path) as f:
            m = json.load(f)
        results[s] = m.get("perplexity_assistant_only_on_common_val") or m.get("perplexity_assistant_only")
    except FileNotFoundError:
        results[s] = None

print("Сводная таблица PPL_assistant_only_on_common_val")
print("=" * 72)
baseline_name = "baseline_masked_packed_base_90k_1ep_noleak"
baseline_ppl = results.get(baseline_name)
for name, ppl in results.items():
    if ppl is None:
        print(f"{name:46s}  —")
        continue
    if baseline_ppl is None or name == baseline_name:
        marker = "(baseline)" if name == baseline_name else ""
    else:
        delta = ppl - baseline_ppl
        pct = delta / baseline_ppl * 100
        marker = f"Δ={delta:+.4f} ({pct:+.2f}%)"
    print(f"{name:46s}  {ppl:.4f}  {marker}")

try:
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    metrics["comparison_strategies"] = {
        "perplexity_assistant_only_on_common_val": results,
        "baseline": baseline_name,
        "baseline_ppl_used_for_delta": baseline_ppl,
    }
    with open(METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"\nОбновлено: {METRICS_PATH} (comparison_strategies)")
except FileNotFoundError:
    print(f"Нет {METRICS_PATH} — пропускаем запись comparison_strategies.")


## (Опционально) Сохранение результатов между сессиями

Сессия Colab умрёт через несколько часов — всё в `/content/` пропадёт. Рекомендуемая схема:

| Что | Куда |
|---|---|
| LoRA-адаптер (~30 МБ) | HuggingFace Hub (приватный репо) |
| `training_log.json`, `final_metrics.json` | Скачать локально и закоммитить в git |

Альтернатива — смонтировать Google Drive и скопировать туда `outputs/{STRATEGY}/` целиком. Если у тебя нет HF-аккаунта или не хочется заводить токен — следующие две ячейки можно пропустить.

Токен берём через `getpass`, чтобы он не попал в git.

In [ ]:
# Раскомментируй и впиши свой username + желаемое имя репо.
# По умолчанию имя репо собирается из STRATEGY — заменишь только username.
# HF_REPO_ID = f"your-username/qwen2.5-1.5b-base-ru-{STRATEGY}"
HF_REPO_ID = None

if HF_REPO_ID is not None:
    from huggingface_hub import HfApi, login
    from getpass import getpass

    login(token=getpass("HF token (write scope): "))

    api = HfApi()
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=ADAPTER_DIR,
        repo_id=HF_REPO_ID,
        commit_message=f"{STRATEGY} adapter ({SUBSAMPLE_SIZE} subsample, {NUM_EPOCHS} epochs, packing=True, train_on_responses_only)",
    )
    api.upload_file(
        path_or_fileobj=METRICS_PATH,
        path_in_repo="final_metrics.json",
        repo_id=HF_REPO_ID,
    )
    api.upload_file(
        path_or_fileobj=LOG_PATH,
        path_in_repo="training_log.json",
        repo_id=HF_REPO_ID,
    )
    print(f"Адаптер и логи запушены в https://huggingface.co/{HF_REPO_ID}")
else:
    print("HF_REPO_ID не задан — пропускаем пуш на Hub.")
    print("Перед закрытием Colab скачай:")
    print(f"  - {ADAPTER_DIR}/")
    print(f"  - {LOG_PATH}")
    print(f"  - {METRICS_PATH}")